# Business entity resolution: features → training → evaluation → test prediction

Runs `scripts/07_features.py` → `08_train.py` → `09_evaluate.py` → `10_predict.py` on Colab.
Step-by-step instructions: **COLAB.md** in the repo (`code/business_entity_resolution/COLAB.md`).

**Runtime:** *Runtime → Change runtime type → CPU* with **High-RAM** on. LightGBM runs on the CPU, so a GPU only
costs compute units. The entity tables plus a training matrix need roughly 15-20 GB; the official
validator on the full `candidate_pairs.tsv` needs up to ~20 GB more.

**Run the cells in order.** Every cell can be re-run. After a disconnect, run all the cells again from the top:
the code is pulled again, the data is copied and verified again (the local disk is wiped when the VM resets),
and stages 07-10 resume from the checkpoints on Google Drive instead of starting over.

Expected times on a High-RAM CPU runtime: data copy ~5-10 min; 07 train ~15 min, val ~5 min, test ~2-2.5 h;
08 ~30-60 min; 09 ~30-60 min (includes two cross-country models); 10 ~20-30 min.

## 0. Settings
Edit these if your Drive folders are named differently.

In [ ]:
import os, subprocess

REPO = "SreeThanu/Business_entity_resolution"     # private GitHub repo
BRANCH = "main"
DRIVE_DATA = "/content/drive/MyDrive/ber_data"     # shortcut to the SHARED data folder (read only)
DRIVE_PERSIST = "/content/drive/MyDrive/ber_persist"  # in YOUR Drive: checkpoints, models, outputs (~10 GB)
REPO_DIR = "/content/ber"
CODE_DIR = f"{REPO_DIR}/code/business_entity_resolution"
LOCAL_DATA = "/content/ber_data"                   # fast local working copy of the data


def run(cmd: str) -> None:
    """Run a shell command in CODE_DIR, stream its output, and stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {cmd}")

## Runtime check
Prints RAM, CPU count and GPU. Recommended: a High-RAM CPU runtime (no GPU needed).

In [ ]:
mem = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
ram_gib = mem["MemTotal"] / 2**20
print(f"RAM: {ram_gib:.1f} GiB total, {mem['MemAvailable'] / 2**20:.1f} GiB available")
print(f"CPUs: {os.cpu_count()}")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if os.path.exists("/usr/bin/nvidia-smi") else None
print("GPU:", gpu.stdout.strip() if gpu and gpu.returncode == 0 else "none (fine: LightGBM runs on the CPU)")
if ram_gib < 40:
    print("\nWARNING: less than 40 GiB RAM. Switch to a High-RAM runtime (Runtime -> Change runtime type).")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(DRIVE_DATA), f"{DRIVE_DATA} not found: add the shared folder as a shortcut in My Drive (COLAB.md, step 1)"
os.makedirs(DRIVE_PERSIST, exist_ok=True)
print("data:", DRIVE_DATA, "| persist:", DRIVE_PERSIST)

## 2. Clone or update the code
Reads the token from Colab Secrets (`GITHUB_TOKEN`, key icon in the left sidebar). The token is sent as an
HTTP header for this one git command only: it is never printed, never stored in the repo's `.git/config`,
and never written to disk.

In [ ]:
import base64
from google.colab import userdata

def git(*args, cwd=None):
    token = userdata.get("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {auth}", *args], cwd=cwd,
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    out = (r.stdout + r.stderr).replace(token, "***").replace(auth, "***")
    if out.strip():
        print(out.strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (output above, token redacted)")

if os.path.isdir(f"{REPO_DIR}/.git"):
    git("fetch", "origin", BRANCH, cwd=REPO_DIR)
    git("checkout", BRANCH, cwd=REPO_DIR)
    git("merge", "--ff-only", f"origin/{BRANCH}", cwd=REPO_DIR)
else:
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 3. Install requirements

In [ ]:
run("pip install -q -r requirements-colab.txt && pip install -q --no-deps -e .")
run("python -c 'import polars, lightgbm, rapidfuzz, numpy; print(\"polars\", polars.__version__, \"| lightgbm\", lightgbm.__version__, \"| rapidfuzz\", rapidfuzz.__version__, \"| numpy\", numpy.__version__)'")

## 4. Copy the data from Drive to the local disk and verify the checksums
Stops here if any file is missing or corrupt (`colab/data_manifest.json`).

In [ ]:
run(f"python scripts/fetch_data.py --src '{DRIVE_DATA}' --dst '{LOCAL_DATA}'")

## 5. Environment variables for stages 07-10

In [ ]:
os.environ.update({
    "BER_ROOT": REPO_DIR,
    "BER_DATA_DIR": LOCAL_DATA,                  # inputs: local disk (fast)
    "BER_PERSIST_DIR": DRIVE_PERSIST,            # checkpoints + models + outputs: Drive (survive a disconnect)
    "BER_OUTPUT_DIR": f"{REPO_DIR}/output",      # submission files (10 also copies them to Drive)
    "BER_MEM_LIMIT_GIB": str(int(ram_gib * 0.85)),  # memory guard: abort cleanly instead of crashing the VM
})
for k in ("BER_ROOT", "BER_DATA_DIR", "BER_PERSIST_DIR", "BER_OUTPUT_DIR", "BER_MEM_LIMIT_GIB"):
    print(f"{k}={os.environ[k]}")

## 6. Pipeline
Each cell resumes from its checkpoints in `BER_PERSIST_DIR`: re-run it after a disconnect (after re-running cells 0-5).

### 07 features (train, val, test)

In [ ]:
run("python scripts/07_features.py --split train")

In [ ]:
run("python scripts/07_features.py --split val")

In [ ]:
run("python scripts/07_features.py --split test")   # ~160M pairs, ~90 chunks; resumes chunk by chunk

### 08 train (LightGBM, early stopping on val)

In [ ]:
run("python scripts/08_train.py")

### 09 evaluate (decision rule, threshold search for macro-F0.5, per country, cross-country)

In [ ]:
run("python scripts/09_evaluate.py")

### 10 predict (test inference, decision rule, submission files, official validator)

In [ ]:
run("python scripts/10_predict.py")

## Outputs
- `ber_persist/output/matching_results.tsv`, `candidate_pairs.tsv`, `predict_summary.json` (Drive)
- `ber_persist/models/<timestamp>_<commit>/`: model, features, params, metrics, `threshold.json`, `eval_report.md`
- `ber_persist/features/`, `ber_persist/preds/`: checkpoints (delete to reclaim space once done)

In [ ]:
run(f"ls -la '{DRIVE_PERSIST}/output' && ls '{DRIVE_PERSIST}/models'")
drive.flush_and_unmount()   # make sure every write reached Drive before the runtime is recycled
print("Drive flushed and unmounted. Re-run cell 1 to mount it again.")